1 Load and look
Load the file into loans_raw and make the copy loans. Print the shape and run info(). Write
down in a comment every problem you can see already: which columns are text that
should be something else, and which column is empty

In [149]:
import pandas as pd

loans_raw = pd.read_csv('loans.csv')
loans = loans_raw.copy()

print(loans.shape)
print(loans.info())

# loan_date/return_date - strings, 
# late_fee - string, 
# non-normalized and inconsistent values,
# missing 'notes'

(623, 9)
<class 'pandas.DataFrame'>
RangeIndex: 623 entries, 0 to 622
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   loan_id      623 non-null    str    
 1   member_id    623 non-null    str    
 2   title        623 non-null    str    
 3   genre        623 non-null    str    
 4   loan_date    623 non-null    str    
 5   return_date  593 non-null    str    
 6   late_fee     623 non-null    str    
 7   returned     623 non-null    str    
 8   notes        0 non-null      float64
dtypes: float64(1), str(8)
memory usage: 84.4 KB
None


2 Clean the genre
How many different values does genre have? It should be six genres. Remove spaces and
capital letters with str.strip() and str.lower() and count again. Look at the values that
are left with unique() and fix the rest with replace. Finish with str.title(). How many
genres do you have, and how many loans in each?

Answer:
genre       loans 
Children    135
Crime       163
Fantasy     90
History     59
Romance     86
Sci-Fi      90

In [150]:
print(loans['genre'].nunique())
print(loans['genre'].value_counts().head(6))

loans['genre'] = (
    loans['genre']
        .str.strip()
        .str.lower()
        .replace('children\'s', 'children')
        .replace('science fiction', 'sci-fi')
        .replace('sci fi', 'sci-fi')
        .str.title()
)

print(loans['genre'].nunique())
print(loans['genre'].value_counts())

print(loans.groupby('genre')['loan_id'].count())

28
genre
Crime         38
Crime         35
crime         33
Romance       32
Children      31
Children's    31
Name: count, dtype: int64
6
genre
Crime       163
Children    135
Fantasy      90
Sci-Fi       90
Romance      86
History      59
Name: count, dtype: int64
genre
Children    135
Crime       163
Fantasy      90
History      59
Romance      86
Sci-Fi       90
Name: loan_id, dtype: int64


3 Yes and no
The column returned says whether the book is back. How many different values does it
have? Turn it into True and False with strip, lower and map. Check that nothing became
missing, and count True and False.

In [151]:
print(loans['returned'].count())
print(loans['returned'].nunique())
print(loans['returned'].value_counts())

loans['returned'] = (
    loans['returned']
        .str.strip()
        .str.lower()
        .map({
            'y': True, 
            'yes': True, 
            'n': False, 
            'no': False}
        )
)

print(loans['returned'].value_counts())


623
10
returned
Y      131
Yes    125
YES    124
y      113
yes    100
N        8
NO       7
No       6
n        6
no       3
Name: count, dtype: int64
returned
True     593
False     30
Name: count, dtype: int64


4 The late fee is text
The late fee looks like 7,50 kr. Turn it into a number. First try astype(float) directly and
read the error. Then remove kr, remove the spaces, change the comma to a dot, and only
then use astype(float). Check the result with describe() and calculate the total of all fees.
On purpose: Try astype(float) on the original text first, and read the error from the bottom
up.

In [152]:
print(loans['late_fee'].dtype) # str
# loans['late_fee'].astype(float) # ValueError: could not convert string to float: '7,50 kr'
loans['late_fee'] = (
    loans['late_fee']
        .str.replace('kr', '')
        .str.replace(' ', '')
        .str.replace(',', '.')

)
loans['late_fee'] = loans['late_fee'].astype(float)
print(loans['late_fee'].describe())
print(loans['late_fee'].sum())

str
count    623.000000
mean      12.323435
std       16.106338
min        0.000000
25%        0.000000
50%        0.000000
75%       25.000000
max       47.500000
Name: late_fee, dtype: float64
7677.5


5 Dates
Turn loan_date into real dates with pd.to_datetime(..., errors='coerce'). How many
dates became NaT? Find the first and the last date. Then count the loans per month with
dt.to_period('M'), and check if the numbers add up to the whole table.

In [153]:
loan_date = loans['loan_date']
print(loan_date.dtype) # str

loan_date = pd.to_datetime(loan_date, errors='coerce')
print(loan_date.dtype) # datetime64[us]

print(loan_date.isna().sum()) # 3
print(loans.loc[loan_date.isna(), ['loan_id', 'loan_date', 'late_fee']]) 
# 477  L-30463
# 483  L-30468
# 592  L-30572
# print(loans.iloc[477]) 

print(loan_date.min()) # 2026-01-05
print(loan_date.max()) # 2026-06-27

loans['loan_date'] = loan_date

loans['loan_date'].dt.to_period('M').value_counts().sort_index() # 620 (623 - 3 NaT)


str
datetime64[us]
3
     loan_id   loan_date  late_fee
477  L-30463     unknown       0.0
483  L-30468  2026-02-30       0.0
592  L-30572     unknown       0.0
2026-01-05 00:00:00
2026-06-27 00:00:00


loan_date
2026-01     96
2026-02    100
2026-03    111
2026-04    106
2026-05    108
2026-06     99
Freq: M, Name: count, dtype: int64

6 Duplicates
How many duplicates are there in the raw table, and how many after the cleaning? Why is
the number different? 
Answer: before cleaning there were extra spaces, dashes, etc.

Check with subset='loan_id'. Remove the duplicates, and reset the
index. How much did the duplicates add to the total of the late fees, in kr and in percent?

In [154]:
print(loans_raw.duplicated().sum()) # 12
print(loans.duplicated().sum()) # 18

print(loans.duplicated(subset='loan_id').sum())
total_fee_before = loans['late_fee'].sum()

loans = loans.drop_duplicates().reset_index(drop=True)

print(loans.duplicated(subset='loan_id').sum())
total_fee_after = loans['late_fee'].sum()

percetage = 100 - total_fee_after * 100 / total_fee_before
print(f'The diff: {total_fee_before - total_fee_after}kr (~{percetage:.1f}%)') # 230.0


12
18
18
0
The diff: 230.0kr (~3.0%)


7 Put it together
Remove the empty column notes. Then collect all your steps in one cell that starts from
loans_raw and gives the finished table. Run it, check the shape, and save the table as
data/loans_clean.csv. Run the cell twice. Does it work both times? Why?

Answer: yes, because all steps are performed from raw csv

In [ ]:
loans = loans.drop(columns='notes')
loans.shape

(605, 8)

In [164]:
import pandas as pd

# 1. Read and copy CSV
loans_raw = pd.read_csv('loans.csv')

loans = loans_raw.copy()
print('Before: ', loans.shape)

# 2. Normalize 'genre'
loans['genre'] = (
    loans['genre']
        .str.strip()
        .str.lower()
        .replace('children\'s', 'children')
        .replace('science fiction', 'sci-fi')
        .replace('sci fi', 'sci-fi')
        .str.title()
)

# 3. Normalize 'returned'
loans['returned'] = (
    loans['returned']
        .str.strip()
        .str.lower()
        .map({
            'y': True, 
            'yes': True, 
            'n': False, 
            'no': False}
        )
)

# 4. Normalize 'late_fee'
loans['late_fee'] = (
    loans['late_fee']
        .str.replace('kr', '')
        .str.replace(' ', '')
        .str.replace(',', '.')

)
loans['late_fee'] = loans['late_fee'].astype(float)

# 5. Convert 'loan_date' to datetime
loan_date = loans['loan_date']
loan_date = pd.to_datetime(loan_date, errors='coerce')
loans['loan_date'] = loan_date

# 6. Drop duplicate rows and empty columns ('notes')
loans = loans.drop_duplicates().reset_index(drop=True)
loans = loans.drop(columns='notes')

# 7. Save cleaned CSV
loans.to_csv('loans_clean.csv', index=False)

print('After: ', loans.shape)

Before:  (623, 9)
After:  (605, 8)
